In [1]:
import sc2ts
import pandas
import numpy as np
import tszip
import matplotlib.pyplot as plt
import pandas as pd
import collections

from pangonet.pangonet import PangoNet

In [2]:
ts = tszip.load("data/sc2ts_viridian_v1.trees.tsz")
ts

In [3]:
ds = sc2ts.Dataset("data/viridian_mafft_2024-10-14_v1.vcz.zip")
ds

2025-08-06 15:43:53,958 INFO:Loading dataset @data/viridian_mafft_2024-10-14_v1.vcz.zip using None as date field


Dataset at data/viridian_mafft_2024-10-14_v1.vcz.zip with 4484157 samples, 29903 variants, and 30 metadata fields. See ds.metadata.field_descriptors() for a description of the fields.

In [4]:
df_node = sc2ts.node_data(ts).set_index("node_id")
df_node

2025-08-06 15:44:00,206 INFO:Computing inheritance counts


,pango,sample_id,scorpio,is_sample,is_recombinant,num_mutations,max_descendant_samples,date
node_id,,,,,,,,
0,B,Vestigial_ignore,.,False,False,0,0,2019-11-17
1,B,Wuhan/Hu-1/2019,.,False,False,0,2482157,2019-12-26
2,A,SRR11772659,.,True,False,1,255,2020-01-19
3,B,SRR11397727,.,True,False,0,1,2020-01-24
4,B,SRR11397730,.,True,False,0,1,2020-01-24
...,...,...,...,...,...,...,...,...
2689049,AY.103,,Delta (B.1.617.2-like),False,False,1,3,2021-11-11
2689050,BA.2.9,,Omicron (BA.2-like),False,False,3,2,2022-03-04
2689051,AY.4,,Delta (AY.4-like),False,False,1,3,2021-09-01


In [5]:
df_samples = df_node[df_node.is_sample]

In [6]:
df_samples.date.max()

Timestamp('2023-02-20 00:00:00')

In [7]:
df_recombinants = pd.read_csv("data/recombinants.csv").set_index("recombinant")
df_recombinants

,sample_id,num_descendant_samples,num_samples,distinct_sample_pango,interval_left,interval_right,num_mutations,Viridian_amplicon_scheme,Artic_primer_version,date_added,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,num_mutations_k1000,num_mutations_k4,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4
recombinant,,,,,,,,,,,,,,,,,,,,,
1530,ERR4437465,1,1,1,8783,13617,2,COVID-ARTIC-V3,3,2020-03-22,...,.,1153.000000,2019-12-26,True,7,2,2,5,5,True
22500,ERR4638271,1,1,1,26528,26714,2,COVID-ARTIC-V3,3,2020-07-23,...,.,1120.002530,2020-01-28,True,6,2,2,11,2,False
26465,ERR4615866,54,1,1,15325,21855,1,COVID-ARTIC-V3,3,2020-08-24,...,.,1120.205275,2020-01-28,True,6,1,2,4,10,True
27003,ERR4671078,3,1,1,22993,25563,2,COVID-ARTIC-V3,3,2020-08-26,...,.,1120.205275,2020-01-28,True,14,2,3,14,10,True
28379,SRR21719160,3,1,1,6542,9515,0,COVID-ARTIC-V3,.,2020-09-02,...,.,1120.205275,2020-01-28,False,4,0,2,5,19,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1430261,ERR10839848,1,1,1,19327,20741,2,COVID-ARTIC-V4.1,4.1alt,2023-01-23,...,Omicron (BA.2-like),452.000000,2021-11-26,True,16,2,11,18,23,True
1430452,SRR23358540,1,1,1,9867,12160,1,COVID-ARTIC-V4.1,.,2023-01-23,...,Omicron (BA.2-like),452.000000,2021-11-26,True,12,1,16,13,30,True
1431988,ERR10931828,1,1,1,19887,21811,2,COVID-ARTIC-V4.1,4.1alt,2023-01-29,...,Omicron (BA.2-like),221.917393,2022-07-15,True,6,2,1,5,4,True


In [8]:
df_pango_x = pd.read_csv("data/bigtable_sc2ts.csv").set_index("pango")
df_pango_x

,github_issue,is_in_results,is_in_arg,is_in_matches,is_clean,is_nested,re_node,type,parent_left_pango,parent_right_pango,parents_extra,interval_left,interval_right,intervals_extra
pango,,,,,,,,,,,,,,
XA,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,False,122444.0,simple,B.1.177.18,B.1.1.7,NaN,20411,21767,NaN
XAA,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,1058654.0,simple,BA.1.1.15,BA.2.9,NaN,4322,5386,NaN
XAC,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,964555.0,simple,BA.2,BA.1.17.2,NaN,24504,26060,NaN
XAD,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,964555.0,simple,BA.2,BA.1.17.2,NaN,24504,26060,NaN
XAE,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,964555.0,simple,BA.2,BA.1.17.2,NaN,24504,26060,NaN
XAF,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,False,False,1177107.0,simple,BA.1.1,BA.2,NaN,10199,10447,NaN
XAG,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,1058654.0,simple,BA.1.1.15,BA.2.9,NaN,4322,5386,NaN
XAJ,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,False,NaN,-1.0,non,NaN,NaN,NaN,NaN,NaN,NaN
XAL,https://github.com/jeromekelleher/sc2ts-paper/...,True,True,NaN,True,True,1003220.0,simple,BA.1.1,BA.2,NaN,17411,21633,NaN


## Compute number of samples in ARG/dataset with descendant pango classification

For each pango, compute the number of samples with pangonet classisifications that are descenedants of it.

In [9]:
pn = PangoNet().build()


2025-08-06 15:44:13,244 INFO:Downloading file: ./alias_key.json
2025-08-06 15:44:13,827 INFO:Downloading file: ./lineage_notes.txt
2025-08-06 15:44:14,662 INFO:Creating aliases.
2025-08-06 15:44:14,667 INFO:Creating network.


In [135]:
df_pango_dataset = ds.metadata.as_dataframe(["Viridian_pangolin_1.29", "Date_tree"])
# FIXME we should also filter this by the sc2ts QC criteria - but this will be tricky to do without 
# actually just running sc2ts. Perhaps the best thing to do is look-up the matches file for 
# each sample.
df_pango_dataset = df_pango_dataset[df_pango_dataset["Date_tree"] <= "2023-02-20"]
df_pango_dataset

,Viridian_pangolin_1.29,Date_tree
sample_id,,
SRR11772659,A,2020-01-19
SRR11597132,A,2020-01-29
SRR12162233,B.1,2020-01-30
SRR12162234,B.1,2020-01-30
SRR11597217,A,2020-02-02
...,...,...
ERR13177412,BQ.1.1,2023-02-15
ERR13177413,XBB.1.5,2023-02-15
ERR13177414,BQ.1.1.4,2023-02-16


# Alternative analysis based on pango X clusters

In [480]:

arg_counts = collections.Counter(df_samples.pango)
ds_counts = collections.Counter(df_pango_dataset["Viridian_pangolin_1.29"])
pango_x_arg_counts = collections.Counter()
pango_x_ds_counts = collections.Counter()
pango_x_descendants = {}
for pango_x in df_pango_x.index.values:
    descendants = pn.get_descendants(pango_x)    
    pango_x_descendants[pango_x] = descendants
    for pango in [pango_x] + descendants:
        pango_x_arg_counts[pango_x] += arg_counts[pango]
        pango_x_ds_counts[pango_x] += ds_counts[pango]
    
    

In [481]:
tree = ts.first()

In [482]:
def analyse_pango_x(pango_x):
   
    df_pango = df_node[df_node.pango.isin([pango_x] + pango_x_descendants[pango_x])]
    #print(pango_x_descendants[pango_x])
    #print(df_pango) 
    events = []
    pango_nodes = set(df_pango.index.values)

    while len(pango_nodes) > 0:
        # Each iteration of this loop corresponds to an event
        descendants = set()
        pairs = sorted([(-ts.nodes_time[u], u) for u in pango_nodes])
        assert len(pairs) == 1 or pairs[0][0] != pairs[1][0]
        first = df_node.loc[pairs[0][1]]
        pango_samples = 0
        non_pango_samples = collections.Counter()
        root = first.name
        for v in tree.nodes(root):
            e = tree.edge(v)
            if ts.edges_left[e] != 0 and ts.edges_right[e] != ts.sequence_length:
                raise ValueError("Recombinant in subgraph")
            descendants.add(v)    

            if tree.is_sample(v):
                if v in pango_nodes:
                    pango_samples += 1
                else:
                    non_pango_samples[df_node.loc[v]["pango"]] += 1

        remaining_pango = pango_nodes - descendants
        non_pango_descendants = descendants - pango_nodes
        #print(df_node.loc[non_pango_descendants]["pango"].value_counts())
        if first.is_sample:
            root_type = "S"
        elif ts.nodes_flags[first.name] & sc2ts.NODE_IS_RECOMBINANT:
            root_type = "R"
        else:
            root_type = "I"
            
        u = root 
        while u != -1 and (ts.nodes_flags[u] & sc2ts.NODE_IS_RECOMBINANT) == 0:
            u = tree.parent(u)
        closest_recombinant = u

        closest_recombinant_path_len = np.inf
        closest_recombinant_time = np.inf
        averted_muts = -1
        if closest_recombinant != -1:
            recomb_record =  df_node.loc[closest_recombinant]
            
            if recomb_record["max_descendant_samples"] < 100_000:    
                closest_recombinant_path_len = tree.path_length(root, closest_recombinant)
                closest_recombinant_time = ts.nodes_time[closest_recombinant] - ts.nodes_time[root] 
                recomb_info = df_recombinants.loc[closest_recombinant]
                averted_muts = recomb_info["num_mutations_k1000"] - recomb_info["num_mutations_k4"]
            else:
                print("Dropping closest recomb", df_node.loc[closest_recombinant].pango)
                closest_recombinant = -1

            
        events.append({
            "pango": pango_x,
            "root": root,
            "root_pango": first.pango,
            "root_mutations": first.num_mutations,
            "root_type": root_type,
            "pango_samples": pango_samples,
            "non_pango_samples": dict(non_pango_samples),
            "arg_count": pango_x_arg_counts[pango_x],
            "ds_count": pango_x_ds_counts[pango_x],
            "closest_recombinant": closest_recombinant,
            "closest_recombinant_path_len": closest_recombinant_path_len,
            "closest_recombinant_time": closest_recombinant_time, 
            "closest_recombinant_averted_mutations": averted_muts,
        })
        
        pango_nodes = remaining_pango
   
    
    return events
e =  analyse_pango_x("XBK") 
pd.DataFrame(e)


Dropping closest recomb BA.2


,pango,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,arg_count,ds_count,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_averted_mutations
0,XBK,1425824,XBK,1,I,7,{},7,157,-1,inf,inf,-1


In [ ]:
analyse_pango_x

In [483]:
data = []
for pango_x in df_pango_x.index.values:
    if pango_x in []: #["XBB", "XM", "XAC", "XAD"]:
        continue
    events = analyse_pango_x(pango_x)
    
    if len(events) > 1:
        events.sort(key=lambda x: -x["pango_samples"])
        print("dropping multi events", pango_x, len(events[1:]))
    data.extend(events[:1])
    
df = pd.DataFrame(data)

dropping multi events XBB 1
Dropping closest recomb BA.5
Dropping closest recomb BA.2
Dropping closest recomb BA.2
Dropping closest recomb BA.5
Dropping closest recomb BA.2
Dropping closest recomb BA.2
Dropping closest recomb BA.5
Dropping closest recomb BA.2
dropping multi events XAC 3
Dropping closest recomb BA.2
Dropping closest recomb BA.5
dropping multi events XM 3
dropping multi events XAD 1


In [440]:
df

,pango,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,arg_count,ds_count,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_averted_mutations
0,XBB,1396207,XBB,14,R,6452,{'BA.2': 1},6454,30882,1396207,0.0,0.000000,8
1,XE,965352,XE,2,S,1116,{},1116,1530,965353,1.0,19.000000,6
2,XBF,1420385,XBF,3,R,185,{},185,705,1420385,0.0,0.000000,6
3,XQ,1058654,XQ,3,R,55,"{'BA.2': 35, 'XAA': 17, 'XU': 1, 'XAM': 21, 'X...",55,76,1058654,0.0,0.000000,7
4,XAZ,1285706,XAZ,3,I,133,{},133,306,-1,inf,inf,-1
5,XN,1061700,XN,2,S,120,{'BA.2': 1},120,203,-1,inf,inf,-1
6,XAS,1265115,XAS,1,S,77,{},77,111,-1,inf,inf,-1
7,XJ,966904,XJ,2,S,68,{'BA.2': 3},68,87,966905,1.0,5.000000,5
8,XBE,1363926,XBE,3,I,65,{},65,109,-1,inf,inf,-1
9,XL,1034619,XL,1,R,64,{},64,84,1034619,0.0,0.000000,8


In [443]:
df = df.sort_values(["closest_recombinant", "closest_recombinant_path_len"], ascending=False)
df["outside"] = df["arg_count"] - df["pango_samples"]


In [449]:
col_name_map = {
    "pango": "pango",
    "root_mutations": "muts",
    "root_type": "type",
    "root": "root",
    "closest_recombinant": "recomb",
    "closest_recombinant_averted_mutations": "averted",
    "closest_recombinant_time": "t recomb",
    "closest_recombinant_path_len": "p recomb",
    "pango_samples": "samples", 
    "outside": "outside",
    "non_pango_samples": "extra" 
    
}

pango_events_tbl = df[list(col_name_map.keys())]
pango_events_tbl

,pango,root_mutations,root_type,root,closest_recombinant,closest_recombinant_averted_mutations,closest_recombinant_time,closest_recombinant_path_len,pango_samples,outside,non_pango_samples
2,XBF,3,R,1420385,1420385,6,0.000000,0.0,185,0,{}
42,XBR,2,R,1420166,1420166,22,0.000000,0.0,1,0,{}
0,XBB,14,R,1396207,1396207,8,0.000000,0.0,6452,2,{'BA.2': 1}
38,XBH,2,I,1395926,1379419,5,35.971066,1.0,2,0,{}
14,XBD,0,R,1378208,1378208,7,0.000000,0.0,30,0,{}
27,XBM,1,I,1358392,1348822,6,23.976698,1.0,10,0,{}
15,XBG,2,R,1291970,1291970,8,0.000000,0.0,25,0,{}
16,XY,2,R,1187989,1187989,5,0.000000,0.0,23,0,{}
39,XAF,9,S,1314603,1177107,5,140.000000,3.0,1,0,{}
13,XW,1,R,1159411,1159411,4,0.000000,0.0,32,0,{}


In [452]:
recombs_tbl = pango_events_tbl[pango_events_tbl["root_type"] == "R"].sort_values(
    "closest_recombinant_averted_mutations", ascending=False)
recombs_tbl

,pango,root_mutations,root_type,root,closest_recombinant,closest_recombinant_averted_mutations,closest_recombinant_time,closest_recombinant_path_len,pango_samples,outside,non_pango_samples
42,XBR,2,R,1420166,1420166,22,0.0,0.0,1,0,{}
34,XC,1,R,414488,414488,14,0.0,0.0,5,0,{}
12,XA,1,R,122444,122444,12,0.0,0.0,39,0,{}
23,XS,1,R,1000242,1000242,11,0.0,0.0,17,0,{}
0,XBB,14,R,1396207,1396207,8,0.0,0.0,6452,2,{'BA.2': 1}
15,XBG,2,R,1291970,1291970,8,0.0,0.0,25,0,{}
9,XL,1,R,1034619,1034619,8,0.0,0.0,64,0,{}
14,XBD,0,R,1378208,1378208,7,0.0,0.0,30,0,{}
3,XQ,3,R,1058654,1058654,7,0.0,0.0,55,0,"{'BA.2': 35, 'XAA': 17, 'XU': 1, 'XAM': 21, 'X..."
2,XBF,3,R,1420385,1420385,6,0.0,0.0,185,0,{}


In [464]:
close_recombs = pango_events_tbl[(pango_events_tbl["root_type"] != "R")
    & (pango_events_tbl["closest_recombinant"] != -1)]
close_recombs = close_recombs.sort_values(["closest_recombinant", "closest_recombinant_time"])
close_recombs

,pango,root_mutations,root_type,root,closest_recombinant,closest_recombinant_averted_mutations,closest_recombinant_time,closest_recombinant_path_len,pango_samples,outside,non_pango_samples
41,XAD,6,S,1108280,964555,6,44.000000,3.0,1,1,{}
10,XZ,2,I,1163537,964555,6,46.424634,3.0,48,0,{}
28,XAE,7,S,1118099,964555,6,47.000000,2.0,9,0,{}
19,XAP,1,I,1216836,964555,6,64.779711,4.0,20,0,{}
30,XAC,3,S,1241084,964555,6,120.000000,4.0,9,9,{}
1,XE,2,S,965352,965353,6,19.000000,1.0,1116,0,{}
37,XH,7,S,1098084,965353,6,55.000000,1.0,2,0,{}
7,XJ,2,S,966904,966905,5,5.000000,1.0,68,0,{'BA.2': 3}
36,XAL,2,I,1264107,1003220,6,63.758451,3.0,3,0,{}
21,XR,4,I,1148222,1058654,7,11.000000,3.0,17,0,{'BA.2': 1}


In [474]:
non_recombs = pango_events_tbl[pango_events_tbl["closest_recombinant"] == -1]
non_recombs = non_recombs.sort_values(["root_mutations", "pango_samples"])
non_recombs

,pango,root_mutations,root_type,root,closest_recombinant,closest_recombinant_averted_mutations,closest_recombinant_time,closest_recombinant_path_len,pango_samples,outside,non_pango_samples
31,XBK,1,I,1425824,-1,-1,inf,inf,7,0,{}
29,XAU,1,S,1231548,-1,-1,inf,inf,8,0,{}
25,XBQ,1,I,1422955,-1,-1,inf,inf,14,0,{}
11,XP,1,S,1092789,-1,-1,inf,inf,45,0,{}
6,XAS,1,S,1265115,-1,-1,inf,inf,77,0,{}
5,XN,2,S,1061700,-1,-1,inf,inf,120,0,{'BA.2': 1}
17,XB,2,I,223239,-1,-1,inf,inf,192,0,{}
26,XAV,3,S,1269391,-1,-1,inf,inf,13,0,{}
8,XBE,3,I,1363926,-1,-1,inf,inf,65,0,{}
4,XAZ,3,I,1285706,-1,-1,inf,inf,133,0,{}


In [484]:
out = []
for tbl in [recombs_tbl, close_recombs, non_recombs]:

    s = tbl.to_latex(
        escape=True, index=False, columns=col_name_map.keys(), header=list(col_name_map.values()),
        float_format="%.0f")
    s = s.replace("'", "").replace(": ", ":")
    s = s.replace("{BA.2:35, XAA:17, XU:1, XAM:21, XAG:6, XR:17}", "{BA.2:35, $\star$}")
    splits = s.splitlines()
    footer = splits[-2:]
    splits = splits[:-2]
    if len(out) > 0:
        splits = splits[3:]
    out.extend(splits)
    

out.extend(footer)
print("\n".join(out))

\begin{tabular}{lrlrrrrrrrl}
\toprule
pango & muts & type & root & recomb & averted & t recomb & p recomb & samples & outside & extra \\
\midrule
XBR & 2 & R & 1420166 & 1420166 & 22 & 0 & 0 & 1 & 0 & {} \\
XC & 1 & R & 414488 & 414488 & 14 & 0 & 0 & 5 & 0 & {} \\
XA & 1 & R & 122444 & 122444 & 12 & 0 & 0 & 39 & 0 & {} \\
XS & 1 & R & 1000242 & 1000242 & 11 & 0 & 0 & 17 & 0 & {} \\
XBB & 14 & R & 1396207 & 1396207 & 8 & 0 & 0 & 6452 & 2 & {BA.2:1} \\
XBG & 2 & R & 1291970 & 1291970 & 8 & 0 & 0 & 25 & 0 & {} \\
XL & 1 & R & 1034619 & 1034619 & 8 & 0 & 0 & 64 & 0 & {} \\
XBD & 0 & R & 1378208 & 1378208 & 7 & 0 & 0 & 30 & 0 & {} \\
XQ & 3 & R & 1058654 & 1058654 & 7 & 0 & 0 & 55 & 0 & {BA.2:35, $\star$} \\
XBF & 3 & R & 1420385 & 1420385 & 6 & 0 & 0 & 185 & 0 & {} \\
XM & 0 & R & 1003220 & 1003220 & 6 & 0 & 0 & 26 & 3 & {BA.2:16, XAL:3} \\
XY & 2 & R & 1187989 & 1187989 & 5 & 0 & 0 & 23 & 0 & {} \\
XF & 1 & R & 946761 & 946761 & 5 & 0 & 0 & 16 & 0 & {} \\
XW & 1 & R & 1159411 & 1159411 & 